# 13 · Embeddings, hardness & active mining at scale

Mining becomes a flywheel: embed frames, index them in **LanceDB**, retrieve the
hard slice's neighbourhood by ANN search, rank by **hardness**, up-weight, retrain.
We run the full cycle on the tiny tier.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
from harness.config import load_config
from harness.train import train
from harness.evaluator import evaluate
from harness.mining import embed
import numpy as np

# 1) a trained model gives us per-frame hardness (low margin = hard)
base = train(load_config("base", {"budget": {"max_epochs": 10}}), DATA)
xtr, ytr, mtr = load_split(DATA, "train")
proba = base.predict_proba(xtr)
margin = np.sort(proba, axis=1)[:, -1] - np.sort(proba, axis=1)[:, -2]
print("hardest frames (lowest margin) by weather:")
order = np.argsort(margin)[:50]
import collections; print(" ", dict(collections.Counter(mtr["weather"][order])))

hardest frames (lowest margin) by weather:
  {'rainy': 16, 'foggy': 23, 'clear': 3, 'overcast': 2, 'snowy': 6}


In [3]:
# 2) index embeddings in LanceDB and expand the fog set by ANN neighbours
import lancedb
feats = embed(xtr).astype("float32")
db = lancedb.connect("/tmp/.mlh_ch13")
try: db.drop_table("f")
except Exception: pass
tbl = db.create_table("f", data=[{"row": i, "weather": str(mtr["weather"][i]),
                                  "vector": feats[i].tolist()} for i in range(len(feats))])
fog = np.where(mtr["weather"] == "foggy")[0]
mined = set(fog.tolist())
for qi in fog[:30]:
    for r in tbl.search(feats[qi].tolist()).limit(6).to_list():
        mined.add(int(r["row"]))
print(f"fog frames: {len(fog)} -> mined neighbourhood: {len(mined)} (LanceDB ANN)")

fog frames: 196 -> mined neighbourhood: 270 (LanceDB ANN)


In [4]:
# 3) up-weight the mined set and retrain; compare the fog slice
w = np.ones(len(xtr)); w[np.fromiter(mined, int)] = 6.0
cfg = load_config("base", {"budget": {"max_epochs": 10}})
mined_model = train(cfg, DATA, sample_weights=w)
b, m = evaluate(base, DATA), evaluate(mined_model, DATA)
print(f"baseline  fog={b['per_weather']['foggy']:.3f}  score={b['score']:.3f}")
print(f"+mining   fog={m['per_weather']['foggy']:.3f}  score={m['score']:.3f}")

baseline  fog=0.689  score=0.928
+mining   fog=0.689  score=0.925


With *unlabeled* frames the same index powers **active learning**: retrieve the
neighbours of hard fog frames and send those for annotation — spend the labeling
budget where the worst slice needs it. Cap the mined fraction per round so the
distribution drifts gradually and another slice doesn't collapse.